# Stage 1. Build the fire libraryTwo halves. The first submits the Earth Engine exports and needs anauthenticated session. The second merges the exported CSV chunks and needsnothing but the files.Skip to the merge if you already have the chunks, or if you are working fromthe archived library.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / "src"))from fireanalogs.config import Configcfg = Config.load(Path.cwd().parent / "config" / "default.yaml")DATA = Path.cwd().parent / cfg["paths"]["data_root"]RESULTS = Path.cwd().parent / cfg["paths"]["results_root"]RESULTS.mkdir(parents=True, exist_ok=True)print("data:", DATA)

## 1a. Earth Engine exportThe inventory is exported in chunks because a single task over the wholeseason exceeds the Earth Engine memory limit. Two summarisers are used. Thefull one produces everything a fire needs to be a leave-one-out target. Theembedding-only one is far cheaper and is run over every event so the retrievalpool is as large as possible.Run this only when regenerating the library from scratch.

In [ ]:
# import ee# ee.Initialize()## from fireanalogs.gee.assets import datasets, fire_collection# from fireanalogs.gee.export import (#     make_summarisers,#     full_export_columns,#     embedding_only_export_columns,#     export_library_chunk,#     wait_for_tasks,# )## ds = datasets(cfg.raw)# fires = fire_collection(cfg.raw)# summarise_full, summarise_embedding_only = make_summarisers(cfg.raw, ds)## EXPORT_FOLDER = "FIRE_EMBEDDINGS"# tasks = {}## # Full summariser, one task per month. Weekly chunks were needed for the# # heaviest months of the fire season.# for year, month in [(2025, 12), (2026, 1)]:#     prefix = f"{year:04d}-{month:02d}"#     chunk = fires.filter(ee.Filter.stringStartsWith("start_date", prefix))#     if chunk.size().getInfo() == 0:#         continue#     name = f"fire_lib_{year}_{month:02d}"#     tasks[name] = export_library_chunk(#         chunk, summarise_full, full_export_columns(cfg.n_emb_dims),#         name, EXPORT_FOLDER,#     )## # Embedding-only summariser over every fire, one task.# tasks["fire_lib_embeddings_only"] = export_library_chunk(#     fires, summarise_embedding_only,#     embedding_only_export_columns(cfg.n_emb_dims),#     "fire_lib_embeddings_only", EXPORT_FOLDER,# )## print(wait_for_tasks(tasks))

## 1b. Merge the exported chunksFires exported with the full summariser can act both as analogs and asvalidation targets. Fires exported with the embedding-only summariser enlargethe retrieval pool without becoming targets. Where a fire appears in both, thefull record wins.

In [ ]:
from fireanalogs.library import load_librarylibrary = load_library(DATA, n_emb_dims=cfg.n_emb_dims)print(f"fires in library          : {len(library)}")print(f"with fuel by slope data   : {int(library['has_fuelslope'].sum())}")print(f"embedding only            : {int((~library['has_fuelslope']).sum())}")print(f"valid observed fractions  : {int(library['obs_high_frac'].notna().sum())}")print(f"date range                : {library['event_date'].min().date()} to {library['event_date'].max().date()}")library.to_csv(RESULTS / "fire_library_merged.csv", index=False)